<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/06%20-%20Quantificadores%20e%20Predicados%20em%20Redes%20de%20Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Quantificadores e Predicados em Redes de Sensores
Validação de integridade e alarme da instrumentação via Lógica de Primeira Ordem (∀ e ∃) em múltiplos cenários de teste.

In [1]:
from typing import Dict, List, Callable

# 1. Definição dos Predicados Lógicos
def ativo(s: Dict) -> bool:
    return s["ativo"]

def falha(s: Dict) -> bool:
    return s["corrente_mA"] < 4.0 or s["corrente_mA"] > 20.0

def critico(s: Dict) -> bool:
    if falha(s):
        return False
    return s["valor"] < s["min_safe"] or s["valor"] > s["max_safe"]

# 2. Implementação dos Quantificadores
def forall(universo: List[Dict], predicado: Callable[[Dict], bool]) -> bool:
    return all(predicado(x) for x in universo)

def exists(universo: List[Dict], predicado: Callable[[Dict], bool]) -> bool:
    return any(predicado(x) for x in universo)

# 3. Matriz de Cenários de Teste
cenarios = {
    "1. Operação Normal": [
        {"tag": "PT-101", "ativo": True, "corrente_mA": 12.0, "valor": 2.1, "min_safe": 0.5, "max_safe": 5.0},
        {"tag": "TT-101", "ativo": True, "corrente_mA": 16.5, "valor": 78.5, "min_safe": 10.0, "max_safe": 85.0},
        {"tag": "AT-101", "ativo": True, "corrente_mA": 14.0, "valor": 6.8, "min_safe": 6.0, "max_safe": 8.0},
        {"tag": "LT-101", "ativo": True, "corrente_mA": 18.2, "valor": 50.0, "min_safe": 10.0, "max_safe": 90.0},
        {"tag": "FT-101", "ativo": True, "corrente_mA": 12.0, "valor": 15.0, "min_safe": 2.0, "max_safe": 40.0}
    ],
    "2. Alarme de Processo": [
        {"tag": "PT-101", "ativo": True, "corrente_mA": 12.0, "valor": 2.1, "min_safe": 0.5, "max_safe": 5.0},
        {"tag": "TT-101", "ativo": True, "corrente_mA": 16.5, "valor": 78.5, "min_safe": 10.0, "max_safe": 85.0},
        {"tag": "AT-101", "ativo": True, "corrente_mA": 14.0, "valor": 6.8, "min_safe": 6.0, "max_safe": 8.0},
        {"tag": "LT-101", "ativo": True, "corrente_mA": 18.2, "valor": 92.0, "min_safe": 10.0, "max_safe": 90.0},
        {"tag": "FT-101", "ativo": True, "corrente_mA": 12.0, "valor": 15.0, "min_safe": 2.0, "max_safe": 40.0}
    ],
    "3. Falha de Sensor": [
        {"tag": "PT-101", "ativo": True, "corrente_mA": 12.0, "valor": 2.1, "min_safe": 0.5, "max_safe": 5.0},
        {"tag": "TT-101", "ativo": True, "corrente_mA": 16.5, "valor": 78.5, "min_safe": 10.0, "max_safe": 85.0},
        {"tag": "AT-101", "ativo": True, "corrente_mA": 14.0, "valor": 6.8, "min_safe": 6.0, "max_safe": 8.0},
        {"tag": "LT-101", "ativo": True, "corrente_mA": 18.2, "valor": 50.0, "min_safe": 10.0, "max_safe": 90.0},
        {"tag": "FT-101", "ativo": True, "corrente_mA": 3.8,  "valor": 0.0,  "min_safe": 2.0, "max_safe": 40.0}
    ]
}

# 4. Avaliação Lógica e Impressão da Tabela de Resultados
print(f"{'Cenário':<22} | {'Rede Íntegra':<13} | {'Alarme Ativo':<12} | {'Ação SIS (Trip)':<15}")
print("-" * 70)

for nome_cenario, lista_sensores in cenarios.items():
    rede_integra = forall(lista_sensores, lambda s: ativo(s) and not falha(s))
    existe_falha = exists(lista_sensores, falha)
    existe_alarme = exists(lista_sensores, lambda s: ativo(s) and critico(s))
    trip_sis = existe_falha or existe_alarme
    
    print(f"{nome_cenario:<22} | {str(rede_integra):<13} | {str(existe_alarme):<12} | {str(trip_sis):<15}")


Cenário                | Rede Íntegra  | Alarme Ativo | Ação SIS (Trip)
----------------------------------------------------------------------
1. Operação Normal     | True          | False        | False          
2. Alarme de Processo  | True          | True         | True           
3. Falha de Sensor     | False         | False        | True           
